In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Retrieval Showdown — BM25 vs Dense vs Hybrid

## 1. Project Overview

**Task:** Compare three retrieval strategies — BM25 (sparse/lexical), Dense (embedding-based), and Hybrid (BM25 + Dense fusion) — on the same question-answering dataset, using identical evaluation metrics so the comparison is fair.

**Why this matters:** Choosing the right retrieval strategy is the single biggest lever in a RAG system. Dense retrieval dominates the hype cycle, but BM25 is surprisingly hard to beat on many tasks. Hybrid methods often win by combining both.

**Stack:**
- `rank_bm25` — BM25 (Okapi BM25) sparse retrieval
- `HuggingFaceEmbeddings` (`all-MiniLM-L6-v2`) — dense retrieval
- `ChromaDB` — vector store for dense retrieval
- `ChatOllama` + `qwen3.5:9b` — LLM for answer generation and judging

> **No API keys required.** Everything runs locally.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | **BM25 mechanics** — how term frequency, inverse document frequency, and document length normalization work |
| 2 | **Dense retrieval** — embedding similarity for semantic matching beyond exact keywords |
| 3 | **Hybrid fusion** — combining sparse and dense scores via Reciprocal Rank Fusion (RRF) |
| 4 | **Fair evaluation** — Recall@k, MRR, and per-query analysis on the same ground-truth set |
| 5 | **Qualitative comparison** — where each method wins and why |
| 6 | **When to use what** — practical guidelines for choosing retrieval strategy |

## 3. Problem Statement

When building a RAG system, you must choose a retrieval method. The three main options:

| Method | How It Works | Strengths | Weaknesses |
|--------|-------------|-----------|------------|
| **BM25** (sparse) | Exact keyword matching with TF-IDF-like scoring | Great for exact terms, names, codes | Misses synonyms and paraphrases |
| **Dense** (embeddings) | Semantic similarity via vector distance | Understands meaning, handles paraphrases | Can miss exact keywords; needs GPU for speed |
| **Hybrid** (BM25 + Dense) | Fuse scores from both methods | Best of both worlds | More complex; requires score normalization |

This notebook runs all three on the **same dataset** and **same queries** so you can see exactly where each excels or fails.

## 4. How Each Method Works

### BM25 (Best Match 25)

BM25 scores a document `d` for query `q` as:

```
score(q, d) = Σ  IDF(t) * [ f(t,d) * (k1 + 1) ] / [ f(t,d) + k1 * (1 - b + b * |d|/avgdl) ]
             t∈q
```

Where:
- `f(t,d)` = frequency of term `t` in document `d`
- `IDF(t)` = inverse document frequency (rare terms score higher)
- `|d|/avgdl` = document length relative to average (penalizes long docs)
- `k1` (default 1.5) and `b` (default 0.75) are tuning parameters

### Dense Retrieval

Encode both query and documents into dense vectors using a transformer model. Retrieve by cosine similarity:

```
similarity(q, d) = cos(embed(q), embed(d)) = (q · d) / (||q|| * ||d||)
```

The embedding model maps semantically similar text to nearby vectors, so "automobile" matches "car" even without shared words.

### Hybrid with Reciprocal Rank Fusion (RRF)

RRF combines ranked lists from multiple retrievers without needing score normalization:

```
RRF_score(d) = Σ  1 / (k + rank_i(d))
              i∈retrievers
```

Where `k` is a constant (typically 60) and `rank_i(d)` is the rank of document `d` in retriever `i`'s results. Documents ranked high by both retrievers get the highest fused score.

## 5. Pipeline Architecture

```
QA Dataset (passages + questions + ground-truth answers)
       │
  ┌────┴──────────────────────────────────────────────┐
  │  INDEXING (one-time)                               │
  │  • BM25: tokenize → build inverted index           │
  │  • Dense: embed → store in ChromaDB                │
  └───┬────────────────┬────────────────┬──────────────┘
      │                │                │
  ┌───┴───┐       ┌───┴───┐       ┌───┴────────┐
  │ BM25  │       │ Dense │       │  Hybrid    │
  │ top-k │       │ top-k │       │ RRF(BM25,  │
  │       │       │       │       │    Dense)  │
  └───┬───┘       └───┬───┘       └───┬────────┘
      │                │                │
  ┌───┴────────────────┴────────────────┴──────────────┐
  │  EVALUATION                                        │
  │  • Recall@k, MRR (same ground-truth set)           │
  │  • Per-query comparison: which method won?          │
  │  • Qualitative analysis: BM25 wins vs Dense wins   │
  │  • LLM answer generation + quality scoring          │
  └────────────────────────────────────────────────────┘
```

## 6. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core langchain-community \
#     langchain-huggingface chromadb sentence-transformers rank_bm25

print("Dependencies: langchain, chromadb, sentence-transformers, rank_bm25")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 7. Imports

## 8. Configuration

In [ ]:
import os
import re
import json
import textwrap
import warnings
from collections import Counter, defaultdict
from typing import Optional

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
from rank_bm25 import BM25Okapi
import chromadb

LLM_MODEL = "qwen3.5:9b"
EMBED_MODEL = "all-MiniLM-L6-v2"
TOP_K = 5
RRF_K = 60  # RRF constant
TEMP_ANSWER = 0.1
TEMP_JUDGE = 0.0

print("All imports OK")
print(f"LLM: {LLM_MODEL}")
print(f"Embeddings: {EMBED_MODEL}")
print(f"BM25: rank_bm25 (Okapi BM25)")
print(f"Retrieval top-k: {TOP_K}")
print(f"RRF constant k: {RRF_K}")

## 9. Helper Functions

In [ ]:
def clean(text: str) -> str:
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.1) -> str:
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    response = llm.invoke(messages)
    return clean(response.content)


def wrap_print(text: str, width: int = 96):
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


# Quick LLM test
test = ask("Say 'ready'.")
print(f"LLM ready: {test[:30]}")

---

# Part A — QA Dataset

## 10. Build the Passage Corpus

We create a synthetic knowledge base of 30 passages covering technology, science, and history. Each passage has an ID and a topic tag. The queries are designed to test both lexical (exact keyword) and semantic (paraphrase) matching.

In [ ]:
PASSAGES = [
    # --- Technology ---
    {"id": "P01", "topic": "technology",
     "text": "Docker containers package applications with their dependencies into isolated units. Unlike virtual machines, containers share the host OS kernel, making them lightweight (megabytes vs gigabytes) and fast to start (seconds vs minutes). Docker images are built from Dockerfiles using layered file systems, where each instruction creates a new layer. Container orchestration tools like Kubernetes manage deployment, scaling, and networking of many containers across clusters."},
    {"id": "P02", "topic": "technology",
     "text": "REST APIs use HTTP methods to perform operations on resources. GET retrieves data, POST creates new resources, PUT updates existing ones, and DELETE removes them. RESTful design emphasizes stateless communication — each request contains all information needed to process it. APIs return standard HTTP status codes: 200 for success, 404 for not found, 500 for server errors. JSON is the dominant data format for modern REST APIs."},
    {"id": "P03", "topic": "technology",
     "text": "Git version control tracks changes to source code over time. Developers create branches to work on features independently, then merge them back to the main branch via pull requests. Git uses a distributed model — every developer has a full copy of the repository history. Common commands include git commit (save changes), git push (upload to remote), git pull (download updates), and git merge (combine branches). Conflicts arise when two branches modify the same lines."},
    {"id": "P04", "topic": "technology",
     "text": "SQL databases organize data in tables with rows and columns. Primary keys uniquely identify each row. Foreign keys create relationships between tables. The four main operations are SELECT (query), INSERT (add), UPDATE (modify), and DELETE (remove). JOINs combine data from multiple tables. Indexes speed up queries by creating sorted references to rows. ACID properties (Atomicity, Consistency, Isolation, Durability) ensure reliable transactions."},
    {"id": "P05", "topic": "technology",
     "text": "The TCP/IP protocol stack has four layers. The application layer (HTTP, FTP, SMTP) provides user-facing services. The transport layer (TCP, UDP) handles reliable delivery — TCP guarantees order and completeness via handshakes and acknowledgments, while UDP is faster but unreliable. The internet layer (IP) handles routing packets across networks using IP addresses. The link layer handles physical transmission over Ethernet, Wi-Fi, or other media."},
    {"id": "P06", "topic": "technology",
     "text": "Machine learning models can be deployed as microservices behind load balancers. The typical architecture includes a model server (Flask, FastAPI, or TensorFlow Serving), a message queue (RabbitMQ or Kafka) for async predictions, a model registry (MLflow) for version management, and monitoring (Prometheus + Grafana) for tracking prediction latency and data drift. Canary deployments route a small percentage of traffic to the new model version before full rollout."},
    # --- Science ---
    {"id": "P07", "topic": "science",
     "text": "Photosynthesis converts solar energy into chemical energy in plants. In the light-dependent reactions (thylakoid membranes), chlorophyll absorbs sunlight and splits water molecules, producing ATP and NADPH while releasing oxygen. In the Calvin cycle (stroma), CO2 from the atmosphere is fixed into glucose using the ATP and NADPH from the light reactions. The overall equation is: 6CO2 + 6H2O + light → C6H12O6 + 6O2."},
    {"id": "P08", "topic": "science",
     "text": "DNA replication is semi-conservative — each new double helix contains one original and one new strand. Helicase unwinds the double helix at the replication fork. Primase adds RNA primers. DNA polymerase III synthesizes the new strand in the 5' to 3' direction. The leading strand is synthesized continuously, while the lagging strand is built in short Okazaki fragments. DNA ligase seals the gaps between fragments. Errors are corrected by proofreading mechanisms."},
    {"id": "P09", "topic": "science",
     "text": "Newton's three laws of motion describe how objects move. The first law (inertia): an object at rest stays at rest, and an object in motion stays in motion at constant velocity, unless acted upon by a net external force. The second law: F = ma, force equals mass times acceleration. The third law: for every action, there is an equal and opposite reaction. These laws form the foundation of classical mechanics and apply to everyday-scale objects."},
    {"id": "P10", "topic": "science",
     "text": "The periodic table organizes elements by atomic number. Elements in the same group (column) share chemical properties because they have the same number of valence electrons. Periods (rows) represent energy levels. Metals are on the left, nonmetals on the right, metalloids along the boundary. Electronegativity generally increases across a period and decreases down a group. Noble gases (Group 18) have full electron shells and are largely unreactive."},
    {"id": "P11", "topic": "science",
     "text": "Plate tectonics explains how Earth's lithosphere is divided into plates that float on the asthenosphere. Divergent boundaries occur where plates move apart (mid-ocean ridges), creating new crust. Convergent boundaries occur where plates collide — oceanic plates subduct under continental plates, forming trenches and volcanic arcs. Transform boundaries involve plates sliding past each other (San Andreas Fault). Plate movement drives earthquakes, volcanic eruptions, and mountain formation."},
    {"id": "P12", "topic": "science",
     "text": "The human immune system has two main branches. Innate immunity provides immediate, non-specific defense through barriers (skin, mucus), phagocytes (neutrophils, macrophages), and inflammation. Adaptive immunity develops targeted responses using lymphocytes: B cells produce antibodies that neutralize specific pathogens, while T cells destroy infected cells directly. Memory cells from adaptive immunity enable faster responses to previously encountered pathogens — the basis of vaccination."},
    # --- History ---
    {"id": "P13", "topic": "history",
     "text": "The Industrial Revolution (1760-1840) transformed manufacturing from hand production to machine-based processes. Key inventions include James Watt's improved steam engine (1769), the spinning jenny for textile production, and the power loom. Factory systems replaced cottage industries, concentrating workers in urban centers. The revolution started in Britain due to abundant coal and iron, a strong patent system, colonial markets, and a mobile labor force displaced from agriculture by the Enclosure Acts."},
    {"id": "P14", "topic": "history",
     "text": "The French Revolution (1789-1799) overthrew the monarchy and established a republic based on Enlightenment ideals. Key causes included financial crisis from war debts, food shortages, social inequality under the estates system, and the influence of philosophers like Rousseau and Voltaire. Major events: the storming of the Bastille (July 14, 1789), the Declaration of the Rights of Man, the Reign of Terror under Robespierre, and Napoleon's rise to power in 1799."},
    {"id": "P15", "topic": "history",
     "text": "The Roman Republic (509-27 BCE) was governed by elected magistrates and the Senate. Two consuls shared executive power with one-year terms to prevent tyranny. The Tribune of the Plebs could veto Senate decisions, protecting common citizens. Roman law influenced modern legal systems — concepts like innocent until proven guilty, the right to a trial, and codified statutes. The Republic collapsed due to military strongmen (Marius, Sulla, Caesar) who accumulated personal power."},
    {"id": "P16", "topic": "history",
     "text": "The Silk Road was a network of trade routes connecting China to the Mediterranean from the 2nd century BCE to the 15th century CE. Besides silk, goods traded included spices, precious metals, glass, and paper. The routes also transmitted religions (Buddhism, Islam, Christianity), technologies (gunpowder, compass, printing), and diseases (Black Death). Caravanserais provided rest stops for merchants. Maritime routes through the Indian Ocean eventually superseded overland trade."},
    {"id": "P17", "topic": "history",
     "text": "The Renaissance (14th-17th century) was a cultural movement originating in Italian city-states like Florence, Venice, and Rome. It emphasized humanism — the study of classical Greek and Roman texts, individual achievement, and secular learning alongside religion. Key figures include Leonardo da Vinci (art and engineering), Michelangelo (sculpture and painting), Galileo (astronomy), and Machiavelli (political philosophy). The invention of the printing press (Gutenberg, 1440) accelerated the spread of Renaissance ideas."},
    {"id": "P18", "topic": "history",
     "text": "The Cold War (1947-1991) was a geopolitical rivalry between the United States and the Soviet Union. It featured an arms race (nuclear weapons), proxy wars (Korea, Vietnam, Afghanistan), space competition (Sputnik, Apollo 11), and ideological conflict (capitalism vs communism). The Berlin Wall (1961-1989) symbolized the divide. Key events include the Cuban Missile Crisis (1962), détente in the 1970s, and the collapse of the Soviet Union in 1991 following reforms by Gorbachev."},
    # --- Additional passages for density ---
    {"id": "P19", "topic": "technology",
     "text": "Cryptographic hash functions map arbitrary data to fixed-size digests. SHA-256 produces a 256-bit hash. Properties: deterministic (same input → same hash), one-way (cannot reverse the hash to get input), collision-resistant (practically impossible to find two inputs with the same hash), and avalanche effect (small input change → completely different hash). Uses include password storage, digital signatures, blockchain proof-of-work, and data integrity verification."},
    {"id": "P20", "topic": "technology",
     "text": "OAuth 2.0 is an authorization framework that allows third-party applications to access user resources without exposing credentials. The authorization code flow: (1) app redirects user to auth server, (2) user logs in and grants permission, (3) auth server returns an authorization code, (4) app exchanges code for access token, (5) app uses token to call API. Refresh tokens allow obtaining new access tokens without re-authentication. JWT (JSON Web Tokens) are commonly used as access tokens."},
    {"id": "P21", "topic": "science",
     "text": "Black holes form when massive stars collapse after exhausting nuclear fuel. The event horizon is the boundary beyond which nothing — not even light — can escape. Hawking radiation theorizes that black holes slowly evaporate by emitting particles from quantum fluctuations near the event horizon. Supermassive black holes (millions to billions of solar masses) reside at the centers of most galaxies. The first image of a black hole (M87) was captured by the Event Horizon Telescope in 2019."},
    {"id": "P22", "topic": "science",
     "text": "CRISPR-Cas9 is a gene editing technology adapted from bacterial immune defenses. A guide RNA directs the Cas9 enzyme to a specific DNA sequence, where it cuts both strands. The cell's repair mechanisms then either disable the gene (NHEJ pathway) or insert a new sequence (HDR pathway using a template). Applications include treating genetic diseases (sickle cell anemia), developing disease-resistant crops, and creating animal models for research. Ethical concerns include off-target edits and germline modifications."},
    {"id": "P23", "topic": "technology",
     "text": "Kubernetes (K8s) orchestrates containerized applications across clusters of machines. A Pod is the smallest deployable unit — one or more containers sharing network and storage. Deployments manage desired state: specify replicas, and K8s ensures that many pods are always running. Services provide stable network endpoints. Ingress controllers route external traffic. ConfigMaps and Secrets manage configuration. Horizontal Pod Autoscaling adjusts pod count based on CPU or custom metrics."},
    {"id": "P24", "topic": "science",
     "text": "Quantum entanglement occurs when two particles become correlated such that measuring one instantly determines the state of the other, regardless of distance. Einstein called this 'spooky action at a distance.' Bell's theorem (1964) and subsequent experiments proved that no local hidden variable theory can explain entanglement — it is a fundamental feature of quantum mechanics. Applications include quantum computing (qubits), quantum cryptography (key distribution), and quantum teleportation of information."},
    {"id": "P25", "topic": "history",
     "text": "The Mongol Empire (1206-1368) was the largest contiguous land empire in history, stretching from Korea to Eastern Europe. Founded by Genghis Khan, it used superior cavalry tactics, psychological warfare, and a meritocratic military structure. The Yam postal system enabled rapid communication across the empire. The Pax Mongolica facilitated trade along the Silk Road. The empire fragmented into khanates after Kublai Khan's death, including the Yuan Dynasty in China and the Golden Horde in Russia."},
    {"id": "P26", "topic": "technology",
     "text": "WebSocket is a protocol that provides full-duplex communication over a single TCP connection. Unlike HTTP's request-response model, WebSocket allows both client and server to send messages independently after an initial handshake. Use cases include real-time chat, live sports scores, collaborative editing, and financial trading dashboards. The connection starts as HTTP and upgrades to WebSocket via the Upgrade header. Libraries like Socket.IO add features like auto-reconnection and room-based messaging."},
    {"id": "P27", "topic": "science",
     "text": "Evolution by natural selection, proposed by Darwin in 1859, explains how species change over time. Individuals with traits better suited to their environment are more likely to survive and reproduce (differential fitness). Over many generations, beneficial traits become more common in the population. Genetic variation arises from mutations, recombination, and gene flow. Speciation occurs when populations are isolated and accumulate enough genetic differences to become reproductively incompatible."},
    {"id": "P28", "topic": "technology",
     "text": "Redis is an in-memory data structure store used as a database, cache, and message broker. It supports strings, lists, sets, sorted sets, hashes, and streams. Redis achieves sub-millisecond latency because all data is in RAM. Persistence options include RDB snapshots and AOF (Append-Only File) logging. Redis Sentinel provides high availability with automatic failover. Redis Cluster enables horizontal scaling by sharding data across multiple nodes. Common use cases: session storage, rate limiting, leaderboards, and pub/sub messaging."},
    {"id": "P29", "topic": "history",
     "text": "The Scientific Revolution (16th-18th century) replaced Aristotelian natural philosophy with empirical, mathematical approaches. Copernicus proposed heliocentrism (1543). Galileo used telescopes to observe Jupiter's moons and Venus's phases, supporting Copernicus. Kepler discovered elliptical orbits. Newton unified terrestrial and celestial mechanics with his laws of motion and universal gravitation (Principia, 1687). The revolution established the scientific method: observation, hypothesis, experimentation, and revision."},
    {"id": "P30", "topic": "history",
     "text": "The Civil Rights Movement in the United States (1954-1968) fought to end racial segregation and discrimination. Key events include Brown v. Board of Education (1954, ending school segregation), the Montgomery Bus Boycott (1955-1956), the March on Washington (1963, Martin Luther King Jr.'s 'I Have a Dream' speech), the Civil Rights Act of 1964 (banning discrimination in public places), and the Voting Rights Act of 1965. The movement used nonviolent protest, civil disobedience, and legal challenges."},
]

print(f"Total passages: {len(PASSAGES)}")
print(f"Topics: {Counter(p['topic'] for p in PASSAGES)}")
print(f"Avg passage length: {sum(len(p['text'].split()) for p in PASSAGES) // len(PASSAGES)} words")

## 11. Inspect the Corpus

In [ ]:
print("PASSAGE CORPUS")
print("=" * 100)
for p in PASSAGES:
    words = len(p["text"].split())
    preview = p["text"][:70].replace("\n", " ")
    print(f"  {p['id']} [{p['topic']:10s}] {words:>3}w | {preview}...")

---

# Part B — Build All Three Retrievers

## 12. BM25 Index

In [ ]:
def tokenize(text: str) -> list[str]:
    """Simple whitespace + lowercase tokenizer for BM25."""
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text.lower())
    return text.split()


corpus_tokens = [tokenize(p["text"]) for p in PASSAGES]
bm25_index = BM25Okapi(corpus_tokens)

print(f"BM25 index built: {len(corpus_tokens)} documents")
print(f"Avg tokens per doc: {sum(len(t) for t in corpus_tokens) // len(corpus_tokens)}")
print(f"Vocabulary size: {len(set(tok for doc in corpus_tokens for tok in doc))}")

## 13. Dense (Embedding) Index

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("passages")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="passages",
    metadata={"hnsw:space": "cosine"},
)

passage_texts = [p["text"] for p in PASSAGES]
passage_metadatas = [{"passage_id": p["id"], "topic": p["topic"]} for p in PASSAGES]
passage_ids = [p["id"] for p in PASSAGES]

vectors = embeddings.embed_documents(passage_texts)
collection.add(
    documents=passage_texts,
    embeddings=vectors,
    metadatas=passage_metadatas,
    ids=passage_ids,
)

print(f"Dense index built: {collection.count()} documents in ChromaDB")
print(f"Embedding dim: {len(vectors[0])}")

## 14. Retrieval Functions — BM25, Dense, Hybrid

In [ ]:
def retrieve_bm25(query: str, top_k: int = TOP_K) -> list[dict]:
    """BM25 sparse retrieval."""
    query_tokens = tokenize(query)
    scores = bm25_index.get_scores(query_tokens)
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    results = []
    for idx in top_indices:
        results.append({
            "passage_id": PASSAGES[idx]["id"],
            "text": PASSAGES[idx]["text"],
            "topic": PASSAGES[idx]["topic"],
            "score": float(scores[idx]),
            "method": "bm25",
        })
    return results


def retrieve_dense(query: str, top_k: int = TOP_K) -> list[dict]:
    """Dense embedding retrieval via ChromaDB."""
    query_vector = embeddings.embed_query(query)
    results = collection.query(query_embeddings=[query_vector], n_results=top_k)
    hits = []
    for i in range(len(results["documents"][0])):
        hits.append({
            "passage_id": results["metadatas"][0][i]["passage_id"],
            "text": results["documents"][0][i],
            "topic": results["metadatas"][0][i]["topic"],
            "score": 1 - results["distances"][0][i],  # cosine similarity
            "method": "dense",
        })
    return hits


def retrieve_hybrid(query: str, top_k: int = TOP_K, rrf_k: int = RRF_K) -> list[dict]:
    """Hybrid retrieval using Reciprocal Rank Fusion (RRF)."""
    bm25_results = retrieve_bm25(query, top_k=top_k * 2)  # get more candidates
    dense_results = retrieve_dense(query, top_k=top_k * 2)

    # Compute RRF scores
    rrf_scores = defaultdict(float)
    passage_data = {}

    for rank, hit in enumerate(bm25_results, 1):
        pid = hit["passage_id"]
        rrf_scores[pid] += 1.0 / (rrf_k + rank)
        passage_data[pid] = hit

    for rank, hit in enumerate(dense_results, 1):
        pid = hit["passage_id"]
        rrf_scores[pid] += 1.0 / (rrf_k + rank)
        if pid not in passage_data:
            passage_data[pid] = hit

    # Sort by fused score
    sorted_ids = sorted(rrf_scores.keys(), key=lambda pid: rrf_scores[pid], reverse=True)[:top_k]

    results = []
    for pid in sorted_ids:
        hit = passage_data[pid]
        results.append({
            "passage_id": pid,
            "text": hit["text"],
            "topic": hit["topic"],
            "score": rrf_scores[pid],
            "method": "hybrid",
        })
    return results


print("All three retrievers ready")
print("  - retrieve_bm25(query)")
print("  - retrieve_dense(query)")
print("  - retrieve_hybrid(query)  [RRF fusion]")

## 15. Quick Retrieval Test

In [ ]:
def display_results(results: list[dict], label: str):
    print(f"\n  [{label}]")
    for i, r in enumerate(results, 1):
        preview = r["text"][:55].replace("\n", " ")
        print(f"    {i}. {r['passage_id']} [{r['topic']:10s}] score={r['score']:.4f} | {preview}...")


q = "how does version control work with branches and merging"
print(f"Q: {q}")

display_results(retrieve_bm25(q, top_k=3), "BM25")
display_results(retrieve_dense(q, top_k=3), "Dense")
display_results(retrieve_hybrid(q, top_k=3), "Hybrid")

---

# Part C — Quantitative Evaluation

## 16. Ground-Truth Evaluation Set

Each query has an expected passage ID. Queries are intentionally varied:
- Some use exact terms from the passage (BM25-friendly)
- Some paraphrase concepts without using the exact words (Dense-friendly)
- Some combine both

In [ ]:
EVAL_SET = [
    # Exact keyword queries (BM25 should do well)
    {"query": "Docker containers vs virtual machines kernel sharing",
     "expected_id": "P01", "query_type": "keyword"},
    {"query": "SQL JOIN foreign key primary key",
     "expected_id": "P04", "query_type": "keyword"},
    {"query": "TCP handshake acknowledgment UDP unreliable",
     "expected_id": "P05", "query_type": "keyword"},
    {"query": "SHA-256 hash collision resistant avalanche",
     "expected_id": "P19", "query_type": "keyword"},
    {"query": "DNA polymerase Okazaki fragments ligase replication fork",
     "expected_id": "P08", "query_type": "keyword"},
    {"query": "CRISPR Cas9 guide RNA gene editing",
     "expected_id": "P22", "query_type": "keyword"},
    {"query": "Mongol Empire Genghis Khan Yam postal system",
     "expected_id": "P25", "query_type": "keyword"},

    # Paraphrase queries (Dense should do well)
    {"query": "how do web applications communicate in real time without polling",
     "expected_id": "P26", "query_type": "paraphrase"},
    {"query": "what is the process by which plants turn sunlight into food",
     "expected_id": "P07", "query_type": "paraphrase"},
    {"query": "how do species gradually change over many generations",
     "expected_id": "P27", "query_type": "paraphrase"},
    {"query": "the struggle for equal rights and ending racial discrimination in America",
     "expected_id": "P30", "query_type": "paraphrase"},
    {"query": "how does the body fight off infections and remember past diseases",
     "expected_id": "P12", "query_type": "paraphrase"},
    {"query": "the cultural rebirth in Italian cities emphasizing classical learning",
     "expected_id": "P17", "query_type": "paraphrase"},
    {"query": "allowing third party apps to access your account without giving your password",
     "expected_id": "P20", "query_type": "paraphrase"},

    # Mixed queries (reasonable for both)
    {"query": "how were goods and ideas exchanged between East and West along ancient trade routes",
     "expected_id": "P16", "query_type": "mixed"},
    {"query": "deploying ML models with Flask behind a load balancer with monitoring",
     "expected_id": "P06", "query_type": "mixed"},
    {"query": "what caused the shift from hand manufacturing to machines in Britain",
     "expected_id": "P13", "query_type": "mixed"},
    {"query": "Einstein spooky action at a distance quantum mechanics",
     "expected_id": "P24", "query_type": "mixed"},
    {"query": "Redis in-memory cache sub-millisecond latency data structures",
     "expected_id": "P28", "query_type": "mixed"},
    {"query": "Kubernetes pods deployments services horizontal autoscaling",
     "expected_id": "P23", "query_type": "mixed"},
    {"query": "what is the relationship between force mass and acceleration",
     "expected_id": "P09", "query_type": "mixed"},
]

print(f"Evaluation set: {len(EVAL_SET)} queries")
print(f"Query types: {Counter(e['query_type'] for e in EVAL_SET)}")

## 17. Run Evaluation — All Three Methods

In [ ]:
def evaluate_retriever(retriever_fn, eval_set: list[dict], top_k: int = 5) -> dict:
    """Compute Recall@k and MRR for a retrieval function."""
    hits_at_k = 0
    recip_ranks = []
    details = []

    for item in eval_set:
        results = retriever_fn(item["query"], top_k=top_k)
        retrieved_ids = [r["passage_id"] for r in results]

        found_rank = None
        for rank, pid in enumerate(retrieved_ids, 1):
            if pid == item["expected_id"]:
                found_rank = rank
                break

        if found_rank is not None:
            hits_at_k += 1
            recip_ranks.append(1.0 / found_rank)
        else:
            recip_ranks.append(0.0)

        details.append({
            "query": item["query"],
            "expected": item["expected_id"],
            "query_type": item["query_type"],
            "top_3": retrieved_ids[:3],
            "hit": found_rank is not None,
            "rank": found_rank,
        })

    n = len(eval_set)
    return {
        "recall_at_k": hits_at_k / n,
        "mrr": sum(recip_ranks) / n,
        "n": n,
        "k": top_k,
        "details": details,
    }


eval_bm25 = evaluate_retriever(retrieve_bm25, EVAL_SET, top_k=5)
eval_dense = evaluate_retriever(retrieve_dense, EVAL_SET, top_k=5)
eval_hybrid = evaluate_retriever(retrieve_hybrid, EVAL_SET, top_k=5)

print("OVERALL RETRIEVAL METRICS")
print("=" * 65)
print(f"{'Metric':<20} {'BM25':>12} {'Dense':>12} {'Hybrid':>12}")
print("-" * 65)
print(f"{'Recall@5':<20} {eval_bm25['recall_at_k']:>11.1%} {eval_dense['recall_at_k']:>11.1%} {eval_hybrid['recall_at_k']:>11.1%}")
print(f"{'MRR':<20} {eval_bm25['mrr']:>11.3f} {eval_dense['mrr']:>11.3f} {eval_hybrid['mrr']:>11.3f}")
print(f"{'Queries':<20} {eval_bm25['n']:>12} {eval_dense['n']:>12} {eval_hybrid['n']:>12}")

## 18. Breakdown by Query Type

In [ ]:
def metrics_by_type(details: list[dict]) -> dict:
    """Group Recall and MRR by query_type."""
    by_type = defaultdict(lambda: {"hits": 0, "rrs": [], "n": 0})
    for d in details:
        qt = d["query_type"]
        by_type[qt]["n"] += 1
        if d["hit"]:
            by_type[qt]["hits"] += 1
            by_type[qt]["rrs"].append(1.0 / d["rank"])
        else:
            by_type[qt]["rrs"].append(0.0)
    result = {}
    for qt, data in by_type.items():
        result[qt] = {
            "recall": data["hits"] / data["n"],
            "mrr": sum(data["rrs"]) / data["n"],
            "n": data["n"],
        }
    return result


bm25_by_type = metrics_by_type(eval_bm25["details"])
dense_by_type = metrics_by_type(eval_dense["details"])
hybrid_by_type = metrics_by_type(eval_hybrid["details"])

print("METRICS BY QUERY TYPE")
print("=" * 80)
for qt in ["keyword", "paraphrase", "mixed"]:
    b = bm25_by_type.get(qt, {"recall": 0, "mrr": 0, "n": 0})
    d = dense_by_type.get(qt, {"recall": 0, "mrr": 0, "n": 0})
    h = hybrid_by_type.get(qt, {"recall": 0, "mrr": 0, "n": 0})
    print(f"\n  {qt.upper()} queries (n={b['n']})")
    print(f"    {'':20s} {'BM25':>10} {'Dense':>10} {'Hybrid':>10}")
    print(f"    {'Recall@5':20s} {b['recall']:>9.1%} {d['recall']:>9.1%} {h['recall']:>9.1%}")
    print(f"    {'MRR':20s} {b['mrr']:>9.3f} {d['mrr']:>9.3f} {h['mrr']:>9.3f}")

## 19. Per-Query Winner Analysis

In [ ]:
print("PER-QUERY COMPARISON")
print("=" * 110)

method_wins = {"bm25": 0, "dense": 0, "hybrid": 0, "tie": 0}

for bd, dd, hd in zip(eval_bm25["details"], eval_dense["details"], eval_hybrid["details"]):
    b_rank = bd["rank"] or 999
    d_rank = dd["rank"] or 999
    h_rank = hd["rank"] or 999

    best_rank = min(b_rank, d_rank, h_rank)
    if best_rank == 999:
        winner = "NONE"
    elif b_rank == d_rank == h_rank:
        winner = "TIE "
        method_wins["tie"] += 1
    elif h_rank <= b_rank and h_rank <= d_rank:
        winner = "HYB "
        method_wins["hybrid"] += 1
    elif b_rank < d_rank:
        winner = "BM25"
        method_wins["bm25"] += 1
    else:
        winner = "DENS"
        method_wins["dense"] += 1

    b_s = f"@{bd['rank']}" if bd["hit"] else "MISS"
    d_s = f"@{dd['rank']}" if dd["hit"] else "MISS"
    h_s = f"@{hd['rank']}" if hd["hit"] else "MISS"

    print(f"  [{winner}] BM25={b_s:>5} Dense={d_s:>5} Hybrid={h_s:>5} "
          f"[{bd['query_type']:>9s}] | {bd['query'][:42]}")

print(f"\nWins — BM25: {method_wins['bm25']} | Dense: {method_wins['dense']} "
      f"| Hybrid: {method_wins['hybrid']} | Tie: {method_wins['tie']}")

## 20. Recall at Different k

In [ ]:
print("RECALL AT DIFFERENT K")
print("=" * 70)
print(f"{'k':>3} | {'BM25 R@k':>10} {'BM25 MRR':>10} | {'Dense R@k':>10} {'Dense MRR':>10} | {'Hyb R@k':>10} {'Hyb MRR':>10}")
print("-" * 70)
for k in [1, 3, 5]:
    b = evaluate_retriever(retrieve_bm25, EVAL_SET, top_k=k)
    d = evaluate_retriever(retrieve_dense, EVAL_SET, top_k=k)
    h = evaluate_retriever(retrieve_hybrid, EVAL_SET, top_k=k)
    print(f"  {k} | {b['recall_at_k']:>9.1%} {b['mrr']:>9.3f} | {d['recall_at_k']:>9.1%} {d['mrr']:>9.3f} | {h['recall_at_k']:>9.1%} {h['mrr']:>9.3f}")

---

# Part D — Qualitative Comparison

## 21. Where BM25 Wins

BM25 excels when the query contains the **exact terms** present in the passage — proper nouns, technical terms, acronyms.

In [ ]:
bm25_friendly = [
    "CRISPR Cas9 guide RNA double strand break",
    "OAuth 2.0 authorization code flow refresh token JWT",
    "Robespierre Reign of Terror Bastille Declaration of Rights",
]

print("QUERIES WHERE BM25 HAS AN ADVANTAGE")
print("(Exact technical terms, proper nouns, acronyms)")
print("=" * 90)
for q in bm25_friendly:
    print(f"\nQ: {q}")
    bm = retrieve_bm25(q, top_k=3)
    de = retrieve_dense(q, top_k=3)
    print(f"  BM25 top-1: {bm[0]['passage_id']} (score={bm[0]['score']:.3f}) | {bm[0]['text'][:50]}...")
    print(f"  Dense top-1: {de[0]['passage_id']} (score={de[0]['score']:.3f}) | {de[0]['text'][:50]}...")
    match = "AGREE" if bm[0]["passage_id"] == de[0]["passage_id"] else "DIFFER"
    print(f"  -> {match}")

## 22. Where Dense Wins

Dense retrieval excels when the query **paraphrases** the passage content — different words, same meaning.

In [ ]:
dense_friendly = [
    "how do living things adapt to their surroundings over long periods",
    "a fast in-memory data store used for caching and real-time apps",
    "the rivalry between two superpowers that almost led to nuclear war",
]

print("QUERIES WHERE DENSE HAS AN ADVANTAGE")
print("(Paraphrased concepts, no exact keywords shared)")
print("=" * 90)
for q in dense_friendly:
    print(f"\nQ: {q}")
    bm = retrieve_bm25(q, top_k=3)
    de = retrieve_dense(q, top_k=3)
    print(f"  BM25 top-1: {bm[0]['passage_id']} (score={bm[0]['score']:.3f}) | {bm[0]['text'][:50]}...")
    print(f"  Dense top-1: {de[0]['passage_id']} (score={de[0]['score']:.3f}) | {de[0]['text'][:50]}...")
    match = "AGREE" if bm[0]["passage_id"] == de[0]["passage_id"] else "DIFFER"
    print(f"  -> {match}")

## 23. Where Hybrid Wins

Hybrid shines when the query has **some exact terms AND some paraphrased concepts** — it catches what both methods individually might rank lower.

In [ ]:
hybrid_friendly = [
    "how does Kubernetes manage container deployment and automatic scaling",
    "what drove the transition to factory-based manufacturing in the 1800s Industrial Revolution",
    "REST API design patterns for creating and retrieving web resources",
]

print("QUERIES WHERE HYBRID COMBINES BOTH SIGNALS")
print("(Mix of exact terms + paraphrased concepts)")
print("=" * 90)
for q in hybrid_friendly:
    print(f"\nQ: {q}")
    bm = retrieve_bm25(q, top_k=3)
    de = retrieve_dense(q, top_k=3)
    hy = retrieve_hybrid(q, top_k=3)
    print(f"  BM25 top-1:   {bm[0]['passage_id']} | {bm[0]['text'][:45]}...")
    print(f"  Dense top-1:  {de[0]['passage_id']} | {de[0]['text'][:45]}...")
    print(f"  Hybrid top-1: {hy[0]['passage_id']} | {hy[0]['text'][:45]}...")
    print(f"  Hybrid top-3: {[r['passage_id'] for r in hy[:3]]}")

---

# Part E — RAG Answer Generation

## 24. Answer Generation Pipeline

Use each retriever to feed context to the LLM and compare answer quality.

In [ ]:
QA_SYSTEM = """You are a knowledgeable assistant. Answer the question using ONLY the provided passages.

Rules:
1. Cite passages as [P01], [P02], etc.
2. If the passages don't contain the answer, say so.
3. Be concise and direct."""


def generate_answer(question: str, retriever_fn, method_name: str) -> dict:
    hits = retriever_fn(question, top_k=3)
    context = "\n\n".join(
        f"[{h['passage_id']}] {h['text']}" for h in hits
    )
    prompt = (
        f"Answer this question using the passages below.\n\n"
        f"QUESTION: {question}\n\n"
        f"PASSAGES:\n{context}\n\n"
        "Return ONLY JSON:\n"
        '{"answer": "answer with [PXX] citations", "sources": ["PXX"]}'
    )
    resp = ask(prompt, system=QA_SYSTEM, temperature=TEMP_ANSWER)
    result = parse_json(resp)
    if not result:
        result = {"answer": resp, "sources": [h["passage_id"] for h in hits[:2]]}
    result["method"] = method_name
    result["retrieved"] = [h["passage_id"] for h in hits]
    return result


print("Answer generation pipeline ready")

## 25. Head-to-Head Answer Comparison

In [ ]:
comparison_questions = [
    "How do containers differ from virtual machines and how are they managed at scale?",
    "What is the process by which plants convert sunlight into energy?",
    "How did ancient trade networks connect East and West?",
]

for q in comparison_questions:
    print(f"\n{'='*90}")
    print(f"Q: {q}")
    print(f"{'='*90}")

    for method_name, retriever_fn in [("BM25", retrieve_bm25), ("Dense", retrieve_dense), ("Hybrid", retrieve_hybrid)]:
        result = generate_answer(q, retriever_fn, method_name)
        print(f"\n  [{method_name}] Retrieved: {result['retrieved'][:3]}")
        ans = str(result.get('answer', ''))
        print(f"  Answer: {textwrap.shorten(ans, width=120, placeholder='...')}")

## 26. Answer Quality — LLM-as-Judge

In [ ]:
JUDGE_SYSTEM = "You evaluate QA answers for quality and grounding."

JUDGE_PROMPT = """Evaluate this answer.

QUESTION: {question}
ANSWER: {answer}
RETRIEVAL METHOD: {method}

Score 1-5:
- correctness: factually accurate?
- relevance: directly answers the question?
- citations: properly cites source passages?
- completeness: covers the key points?

Return ONLY JSON:
{{"correctness": N, "relevance": N, "citations": N, "completeness": N,
  "overall": N, "notes": "explanation"}}"""

judge_q = "How do containers differ from virtual machines and how are they managed at scale?"
print(f"Q: {judge_q}")
print("=" * 80)

for method_name, retriever_fn in [("BM25", retrieve_bm25), ("Dense", retrieve_dense), ("Hybrid", retrieve_hybrid)]:
    result = generate_answer(judge_q, retriever_fn, method_name)
    resp = ask(
        JUDGE_PROMPT.format(
            question=judge_q,
            answer=str(result.get("answer", ""))[:400],
            method=method_name,
        ),
        system=JUDGE_SYSTEM,
        temperature=TEMP_JUDGE,
    )
    scores = parse_json(resp)
    if scores:
        print(f"\n  [{method_name}]")
        for dim in ["correctness", "relevance", "citations", "completeness"]:
            val = scores.get(dim, "?")
            bar = "*" * int(val) if isinstance(val, (int, float)) else "?"
            print(f"    {dim:16s}: {val}/5 {bar}")
        print(f"    {'overall':16s}: {scores.get('overall', '?')}/5")

---

# Part F — Analysis & Learning Notes

## 27. Summary of Results

### When to Use Each Method

| Situation | Best Method | Why |
|-----------|------------|-----|
| Exact term matching (code names, IDs, acronyms) | **BM25** | Lexical matching catches exact token overlap |
| Paraphrased / semantic queries | **Dense** | Embeddings capture meaning, not just words |
| General-purpose RAG system | **Hybrid** | Covers both lexical and semantic; robust across query types |
| Low-resource setup (no GPU) | **BM25** | No neural model needed; fast on CPU |
| Small corpus (<1K docs) | **Dense** or **Hybrid** | Embedding plus brute-force cosine is fast enough |
| Large corpus (>1M docs) | **BM25** + reranker | Inverted index scales; use dense model to rerank top-k |

### Key Insight: BM25 Is Not Dead

Despite the hype around dense retrieval, BM25:
- Requires no GPU or model download
- Has zero training cost — it is a pure statistical method
- Handles rare terms, names, and codes better than embeddings
- Scales to billions of documents with inverted indexes

### Key Insight: Hybrid Is Usually the Safest Choice

RRF (Reciprocal Rank Fusion) is elegant because:
- No score normalization needed (rank-based, not score-based)
- A document ranked high by just one method still appears
- A document ranked high by both methods gets a strong boost
- The `k` constant (60) prevents any single rank from dominating

In [ ]:
# Final side-by-side comparison table
print("FINAL COMPARISON TABLE")
print("=" * 65)
print(f"{'':20s} {'BM25':>12} {'Dense':>12} {'Hybrid':>12}")
print("-" * 65)
print(f"{'Recall@5':20s} {eval_bm25['recall_at_k']:>11.1%} {eval_dense['recall_at_k']:>11.1%} {eval_hybrid['recall_at_k']:>11.1%}")
print(f"{'MRR':20s} {eval_bm25['mrr']:>11.3f} {eval_dense['mrr']:>11.3f} {eval_hybrid['mrr']:>11.3f}")

print(f"\n{'By Query Type':20s}")
for qt in ["keyword", "paraphrase", "mixed"]:
    b = bm25_by_type.get(qt, {"recall": 0})
    d = dense_by_type.get(qt, {"recall": 0})
    h = hybrid_by_type.get(qt, {"recall": 0})
    print(f"  {qt:>12s} R@5: {b['recall']:>8.1%} {d['recall']:>11.1%} {h['recall']:>11.1%}")

print(f"\n{'Winner counts':20s}")
print(f"  BM25: {method_wins['bm25']:>2} | Dense: {method_wins['dense']:>2} "
      f"| Hybrid: {method_wins['hybrid']:>2} | Tie: {method_wins['tie']:>2}")
print(f"\n{'Requirements':20s}")
print(f"  {'GPU needed':20s} {'No':>12} {'Yes*':>12} {'Yes*':>12}")
print(f"  {'Index build time':20s} {'Fast':>12} {'Moderate':>12} {'Moderate':>12}")
print(f"  {'Extra deps':20s} {'rank_bm25':>12} {'transformers':>12} {'Both':>12}")
print(f"\n  * Dense runs on CPU too but is slower")

## 28. Common Mistakes

| Mistake | Why It's Wrong | Better Approach |
|---------|---------------|----------------|
| Using only dense retrieval | Misses exact keyword matches; slower | Start with BM25 baseline, add dense if needed |
| Treating BM25 as outdated | It still beats dense on keyword-heavy queries | Always include BM25 in comparison |
| Normalizing scores directly for fusion | BM25 and cosine scores are on different scales | Use rank-based fusion (RRF) instead |
| Not splitting evaluation by query type | Hides where each method fails | Report keyword vs paraphrase performance separately |
| Evaluating only Recall@1 | Too harsh; misses useful results at position 2-5 | Use Recall@k and MRR together |
| Ignoring tokenization for BM25 | Poor tokenization (no lowercasing, no punctuation removal) hurts BM25 | Clean and lowercase text before indexing |

## 29. Production Improvement Ideas

1. **Learned sparse retrieval** — use SPLADE or DeepImpact for neural sparse representations
2. **Cross-encoder reranking** — retrieve top-50 with BM25, rerank with a cross-encoder for precision
3. **Query classification** — detect whether a query is keyword or semantic, route accordingly
4. **Domain-specific embeddings** — fine-tune embedding model on domain data for better dense retrieval
5. **Dynamic RRF weight** — adjust the contribution of BM25 vs dense based on query characteristics
6. **ColBERT** — late interaction model that is both semantically aware and efficient at scale
7. **Negative mining** — train dense retriever with hard negatives from BM25 errors

## 30. Exercises

### Exercise 1
Add a cross-encoder reranker: retrieve top-20 with BM25, then rerank using the embedding model's similarity scores. Compare Recall@5 and MRR against pure BM25 and pure dense.

### Exercise 2
Implement weighted hybrid fusion: instead of equal RRF contribution, give BM25 weight `alpha` and dense weight `1-alpha`. Sweep alpha from 0.0 to 1.0 and plot Recall@5 to find the optimal balance.

### Exercise 3
Build a query classifier: use the LLM to classify each query as "keyword" or "semantic" and route to BM25 or Dense accordingly. Compare this adaptive routing against always-hybrid.

### Mini Challenge
Add a third retriever: TF-IDF with cosine similarity (using scikit-learn). Include it in the RRF fusion and the evaluation. Does three-way fusion beat two-way?

In [ ]:
print("SESSION SUMMARY")
print("=" * 60)
print(f"Passages indexed: {len(PASSAGES)}")
print(f"Topics: {dict(Counter(p['topic'] for p in PASSAGES))}")
print(f"Evaluation queries: {len(EVAL_SET)}")
print(f"Query types: {dict(Counter(e['query_type'] for e in EVAL_SET))}")
print(f"\nRetrieval Recall@5 / MRR:")
print(f"  BM25:   {eval_bm25['recall_at_k']:.1%} / {eval_bm25['mrr']:.3f}")
print(f"  Dense:  {eval_dense['recall_at_k']:.1%} / {eval_dense['mrr']:.3f}")
print(f"  Hybrid: {eval_hybrid['recall_at_k']:.1%} / {eval_hybrid['mrr']:.3f}")
print(f"\nCapabilities built:")
print(f"  - BM25 (Okapi BM25) sparse retrieval")
print(f"  - Dense (MiniLM + ChromaDB) embedding retrieval")
print(f"  - Hybrid (Reciprocal Rank Fusion) combined retrieval")
print(f"  - Metrics: Recall@k, MRR, per-query and per-type breakdown")
print(f"  - Qualitative analysis: where each method wins and why")
print(f"  - RAG answer generation + LLM-as-judge quality scoring")

## 31. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **BM25 is not dead** — it excels on exact keyword queries and requires no GPU |
| 2 | **Dense retrieval captures semantics** — paraphrased queries that share no words still match |
| 3 | **Hybrid (RRF) is the safest default** — it handles both query types by combining both signals |
| 4 | **Evaluate by query type** — overall metrics hide that BM25 wins on keywords, Dense on paraphrases |
| 5 | **RRF avoids score normalization** — rank-based fusion is simpler and more robust than score fusion |
| 6 | **Always establish a BM25 baseline first** — it takes 5 lines of code and sets the bar |
| 7 | **Retrieval quality directly determines answer quality** — if the right passage isn't retrieved, the LLM can't answer |
| 8 | **Qualitative analysis reveals failure modes** — numbers alone don't tell you *why* a method fails |
| 9 | **Per-query comparison** is more informative than aggregate metrics for system design decisions |
| 10 | In production, **start hybrid, then optimize** — measure which query types dominate and tune accordingly |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*